In [2]:
import pandas as pd 


# Loading all 4 CSVs files from DDInter2.0 database
csv_files = [
    "data/ddinter_downloads_code_B.csv",
    "data/ddinter_downloads_code_D.csv",
    "data/ddinter_downloads_code_A.csv",
    "data/ddinter_downloads_code_P.csv",
]

In [6]:
dfs = []

for f in csv_files:
    df = pd.read_csv(f)
    dfs.append(df)
    print(f"{f}:{len(df)} rows, columns: {list(df.columns)}")

# combining
combined = pd.concat(dfs, ignore_index=True)
print(f"Combined: {len(combined)} rows")
print(f"Columns: {list(combined.columns)}")
print(combined.head(3).to_string())

data/ddinter_downloads_code_B.csv:15140 rows, columns: ['DDInterID_A', 'Drug_A', 'DDInterID_B', 'Drug_B', 'Level']
data/ddinter_downloads_code_D.csv:25681 rows, columns: ['DDInterID_A', 'Drug_A', 'DDInterID_B', 'Drug_B', 'Level']
data/ddinter_downloads_code_A.csv:56367 rows, columns: ['DDInterID_A', 'Drug_A', 'DDInterID_B', 'Drug_B', 'Level']
data/ddinter_downloads_code_P.csv:5492 rows, columns: ['DDInterID_A', 'Drug_A', 'DDInterID_B', 'Drug_B', 'Level']
Combined: 102680 rows
Columns: ['DDInterID_A', 'Drug_A', 'DDInterID_B', 'Drug_B', 'Level']
  DDInterID_A        Drug_A DDInterID_B             Drug_B  Level
0  DDInter975          Iron  DDInter582       Dolutegravir  Major
1  DDInter582  Dolutegravir  DDInter725   Ferrous fumarate  Major
2  DDInter582  Dolutegravir  DDInter726  Ferrous gluconate  Major


In [7]:
# extracting unique drugs from both A and B columns
drugs_a = combined[["Drug_A", "DDInterID_A"]].rename(
    columns={"Drug_A": "drug_name", "DDInterID_A": "ddinter_id"})
drugs_b = combined[["Drug_B", "DDInterID_B"]].rename(
    columns={"Drug_B": "drug_name", "DDInterID_B": "ddinter_id"})


# combining and deduplicate
all_drugs = pd.concat([drugs_a, drugs_b], ignore_index=True)
drug_lookup = all_drugs.drop_duplicates(
    subset = ["drug_name"]).sort_values("drug_name").reset_index(drop=True)


print(f"Total unique drugs: {len(drug_lookup)}")
print("Printing first and last 10")
print(drug_lookup.head(10).to_string(index=False))
print("-")
print(drug_lookup.tail(10).to_string(index=False))

Total unique drugs: 1867
Printing first and last 10
           drug_name ddinter_id
            Abacavir   DDInter1
       Abaloparatide   DDInter2
Abametapir (topical)   DDInter3
            Abarelix   DDInter4
           Abatacept   DDInter5
           Abciximab   DDInter6
         Abemaciclib   DDInter7
         Abiraterone   DDInter8
       Acalabrutinib   DDInter9
         Acamprosate  DDInter10
-
              drug_name  ddinter_id
          Zinc chloride DDInter1964
         Zinc gluconate DDInter1965
           Zinc sulfate DDInter1966
            Ziprasidone DDInter1967
        Zoledronic acid DDInter1968
           Zolmitriptan DDInter1969
               Zolpidem DDInter1970
             Zonisamide DDInter1971
              Zopiclone DDInter1972
gamma-Aminobutyric acid  DDInter805


In [8]:
# checking for drugs with multiple DDInterIDs
id_check = all_drugs.drop_duplicates().groupby("drug_name")["ddinter_id"].nunique()
multi_id = id_check[id_check > 1]
if len(multi_id) > 0:
    for name, count in multi_id.items():
        ids = all_drugs[all_drugs["drug_name"] == name]["ddinter_id"].unique()
        print(f"{name}:{ids}")

else:
    print("Every drug has exactly one DDInterID")

Every drug has exactly one DDInterID


In [9]:
# saving csv
drug_lookup.to_csv("ddinter_drug_lookup.csv", index=False)